# 07 — FF5 + Momentum + CG Panel Regression

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import yfinance as yf
import statsmodels.api as sm
import os

In [2]:
BASE = os.getcwd()

RF_ANNUAL  = 0.065
RF_MONTHLY = RF_ANNUAL / 12
RF_DAILY   = RF_ANNUAL / 252

FY_WINDOWS = {
    'FY23': ('2022-04-01', '2023-03-31'),
    'FY24': ('2023-04-01', '2024-03-31'),
    'FY25': ('2024-04-01', '2025-03-31'),
}

Q4_FYS = ['Q4FY22', 'Q4FY23', 'Q4FY24']
FACTOR_COLS = ['Mkt_RF','SMB','HML','RMW','CMA','MOM']
MIN_OBS = 180
FILING_WIN = 252

In [3]:
imap = pd.read_excel(BASE + '/data/processed/industry_map.xlsx')
imap = imap.dropna(subset=['NSE Symbol']).copy()
imap['BSE Code'] = pd.to_numeric(imap['BSE Code'], errors='coerce').astype('Int64')
tickers_ns = [f"{s}.NS" for s in imap['NSE Symbol']]
print(f'Companies: {len(tickers_ns)}')

fdb = pd.read_csv(BASE + '/data/processed/filing_dates_db.csv')
fdb['BSE_Code'] = pd.to_numeric(fdb['BSE_Code'], errors='coerce').astype('Int64')
fdb['Filing_Date'] = pd.to_datetime(fdb['Filing_Date'])

q4_filings = (
    fdb[fdb['Q_FY'].isin(Q4_FYS)]
    [['BSE_Code','Q_FY','Filing_Date']]
    .merge(imap[['BSE Code','NSE Symbol']], left_on='BSE_Code', right_on='BSE Code', how='inner')
    .drop(columns='BSE Code')
    .reset_index(drop=True)
)
q4_filings['FY'] = 'FY' + q4_filings['Q_FY'].str[-2:]

print(f'Q4 filings loaded: {len(q4_filings)} rows')
print(q4_filings.groupby('Q_FY')['NSE Symbol'].count().to_string())


Companies: 247
Q4 filings loaded: 711 rows
Q_FY
Q4FY22    217
Q4FY23    247
Q4FY24    247


In [4]:
px_raw = yf.download(tickers_ns, start='2019-07-01', end = '2026-05-01',
                     auto_adjust = True, progress = True)['Close']

valid = px_raw.columns[px_raw.isna().mean() < 0.5].tolist()
px = px_raw[valid]

print(f'Valid tickers: {len(valid)} / {len(tickers_ns)}')
print(f'Date range: {px.index[0].date()} -> {px.index[-1].date()}')

ret_d = np.log(px / px.shift(1)).dropna(how = 'all')
print('Daily returns shape:', ret_d.shape)

[                       0%                       ]

[                       1%                       ]  3 of 247 completed

[*                      2%                       ]  4 of 247 completed

[*                      2%                       ]  5 of 247 completed

[*                      2%                       ]  6 of 247 completed

...



11 Failed downloads:


['SYRMA.NS', 'TATACONSUM.NS', 'UCOBANK.NS', 'INDIANB.NS', 'GRSE.NS']: TypeError("'NoneType' object is not subscriptable")


['ISEC.NS', 'GET&D.NS', 'ZOMATO.NS', 'SUVENPHARMA.NS', 'PEL.NS']: possibly delisted; no timezone found


['AKZOINDIA.NS']: possibly delisted; no price data found  (1d 2019-07-01 -> 2026-05-01) (Yahoo error = "No data found, symbol may be delisted")


Valid tickers: 233 / 247
Date range: 2019-07-01 -> 2026-04-30
Daily returns shape: (1689, 233)


In [5]:
factors_m = pd.read_csv(BASE + '/data/processed/ff5mom_factors_monthly.csv',
                        parse_dates = ['Date'])

factors_m = factors_m.set_index('Date')

fac_m = factors_m.copy()
fac_m.index = fac_m.index.year * 100 + fac_m.index.month

daily_key = ret_d.index.year * 100 + ret_d.index.month
td_per_month = pd.Series(1, index = daily_key).groupby(level = 0).sum()

fac_for_day = fac_m.reindex(daily_key)
td_for_day = td_per_month.reindex(daily_key).fillna(21).values

factors_d = pd.DataFrame(
    fac_for_day.values/ td_for_day[:, None],
    index = ret_d.index,
    columns = FACTOR_COLS,
)

factors_d['CMA'] = factors_d['CMA'].fillna(0)

print(f'Monthly factors shape: {factors_m.shape}')
print(f'Daily factors shape: {factors_d.shape}')
print('NaN count per column:')
print(factors_d.isna().sum().to_string())

Monthly factors shape: (46, 6)
Daily factors shape: (1689, 6)
NaN count per column:
Mkt_RF    744
SMB       744
HML       744
RMW       744
CMA         0
MOM       744


---
## Section: Augmented FF5+MOM+CG Panel Regression

**Model (pooled panel):**

`(r_it − rf_t) = α + β₁·MktRF_t + β₂·SMB_t + β₃·HML_t + β₄·RMW_t + β₅·CMA_t + β₆·MOM_t + β_CG·CG_Score_i + ε_it`

CG score (composite average score) is **constant within each firm-window** — it varies cross-sectionally.  
Identification comes from cross-sectional variation in CG scores across firms.  
Inference: firm-clustered standard errors.  
Estimated separately for **Annual (FY)** and **Filing-date (Q4)** windows.


In [6]:
from statsmodels.stats.stattools import jarque_bera, durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan

scores_path = BASE + '/data/processed/cg_scores.csv'
scores = pd.read_csv(scores_path)
scores['BSE Code'] = pd.to_numeric(scores['BSE Code'], errors = 'coerce').astype('Int64')

imap = pd.read_excel(BASE + '/data/processed/industry_map.xlsx')[['BSE Code', 'NSE Symbol']]
imap['BSE Code'] = pd.to_numeric(imap['BSE Code'], errors = 'coerce').astype('Int64')
bse_to_sym = dict(zip(imap['BSE Code'], imap['NSE Symbol']))

scores['FY'] = 'FY' + scores['Q_FY'].str[-2:]
cg_ann = (scores.groupby(['BSE Code', 'FY'])['Avg_Score']
          .mean().reset_index().rename(columns={'Avg_Score': 'CG_score'}))
cg_ann['NSE Symbol'] = cg_ann['BSE Code'].map(bse_to_sym)
cg_ann_map = cg_ann.set_index(['NSE Symbol', 'FY'])['CG_score'].to_dict()

cg_q = (scores.groupby(['BSE Code', 'Q_FY'])['Avg_Score']
        .mean().reset_index().rename(columns = {'Avg_Score': 'CG_score'}))
cg_q['NSE Symbol'] = cg_q['BSE Code'].map(bse_to_sym)
cg_q_map = cg_q.set_index(['NSE Symbol', 'Q_FY'])['CG_score'].to_dict()

print(f'Annual CG composite:    {len(cg_ann)} firm-FY pairs, score range: '
      f'{cg_ann.CG_score.min():.2f} – {cg_ann.CG_score.max():.2f}')
print(f'Quarterly CG composite: {len(cg_q)} firm-Q pairs')


def build_panel(windows, ticker_list, factor_daily, cg_lookup, window_type = 'fy'):
    """Stack all firm-day excess returns for the given windows into one DataFrame."""
    rows = []
    if window_type == 'fy':
        for label, (start, end) in windows.items():
            fac_w = factor_daily.loc[start:end, FACTOR_COLS].dropna(
                subset = ['Mkt_RF', 'SMB', 'HML', 'RMW', 'MOM']
            )
            for ticker in ticker_list:
                sym = ticker.replace('.NS', '')
                cg = cg_lookup.get((sym, label), np.nan)
            
                if pd.isna(cg):
                    continue

                stk = ret_d.loc[start:end, ticker].dropna()
                aligned = pd.concat([stk.rename('r'), fac_w], axis=1).dropna()
                
                if len(aligned) < MIN_OBS:
                    continue

                tmp = aligned.copy()
                tmp['excess_r'] = tmp['r'] - RF_DAILY
                tmp['CG_score'] = cg
                tmp['NSE Symbol'] = sym
                tmp['window'] = label
                rows.append(tmp[['excess_r', 'Mkt_RF', 'SMB', 'HML', 'RMW', 'CMA', 'MOM',
                                 'CG_score', 'NSE Symbol', 'window']])
    else:
        for sym, q_fy, filing_dt in windows:
            ticker = sym + '.NS'
            if ticker not in ticker_list:
                continue
            cg = cg_lookup.get((sym, q_fy), np.nan)
            if pd.isna(cg):
                continue
            win = ret_d.index[ret_d.index > filing_dt][:FILING_WIN]
            if len(win) < MIN_OBS:
                continue
            stk = ret_d.loc[win, ticker].dropna()
            fac_w = factor_daily.loc[win, FACTOR_COLS].dropna(
                subset=['Mkt_RF', 'SMB', 'HML', 'RMW', 'MOM']
            )
            aligned = pd.concat([stk.rename('r'), fac_w], axis=1).dropna()
            if len(aligned) < MIN_OBS:
                continue
            tmp = aligned.copy()
            tmp['excess_r'] = tmp['r'] - RF_DAILY
            tmp['CG_score'] = cg
            tmp['NSE Symbol'] = sym
            tmp['window'] = q_fy
            rows.append(tmp[['excess_r', 'Mkt_RF', 'SMB', 'HML', 'RMW', 'CMA', 'MOM',
                             'CG_score', 'NSE Symbol', 'window']])
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def run_panel_ols(panel, title):
    if panel.empty:
        print(f'{title}: empty panel')
        return

    Y = panel['excess_r'].astype(float)
    X_cols = FACTOR_COLS + ['CG_score']
    if panel['window'].nunique() > 1:
        fe = pd.get_dummies(panel['window'], prefix = '_w', drop_first = True)
        X = sm.add_constant(pd.concat([panel[X_cols], fe], axis = 1).astype(float), has_constant = 'add')
    else:
        X = sm.add_constant(panel[X_cols].astype(float), has_constant = 'add')

    base = sm.OLS(Y, X).fit()
    try:
        res = base.get_robustcov_results(cov_type = 'cluster', groups = panel['NSE Symbol'].values)
    except Exception:
        res = base.get_robustcov_results('HC3')

    xnames = list(X.columns)
    params = pd.Series(np.asarray(res.params), index = xnames)
    bse = pd.Series(np.asarray(res.bse), index = xnames)
    pvals = pd.Series(np.asarray(res.pvalues), index = xnames)
    tvals = pd.Series(np.asarray(res.tvalues), index = xnames)

    def stars(p):
        return '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else ''

    bar = '─' * 60
    print(f'\n{"═" * 60}')
    print(f'  {title}')
    print(f'  N firms: {panel["NSE Symbol"].nunique()}  |  '
          f'N obs: {len(panel):,}  |  R²={base.rsquared:.4f}  |  '
          f'Adj-R²={base.rsquared_adj:.4f}')
    print(f'{"═" * 60}')
    print(f'  {"Variable":<16} {"Coef":>10} {"SE":>10} {"t":>8} {"p":>8}  ')
    print(f'  {bar}')
    for v in xnames:
        if v.startswith('_w') or v == 'const':
            continue
        marker = ' ◀' if v == 'CG_score' else ''
        print(f'  {v:<16} {params[v]:>10.6f} {bse[v]:>10.6f} '
              f'{tvals[v]:>8.3f} {pvals[v]:>8.4f} {stars(pvals[v])}{marker}')
    print(f'  {bar}')
    print(f'  {"const":<16} {params.get("const", np.nan):>10.6f}')
    if panel['window'].nunique() > 1:
        print(f'  Window FE: Yes ({panel["window"].nunique()} windows)')
    print(f'{"═" * 60}')

    resid = base.resid.values
    jb_p = jarque_bera(resid)[1]
    dw = durbin_watson(resid)
    try:
        _, bp_p, _, _ = het_breuschpagan(resid, X)
    except Exception:
        bp_p = np.nan
    print(f'  JB p={jb_p:.4f}  |  BP p={bp_p:.4f}  |  DW={dw:.3f}')
    print('  (clustered SE corrects for heteroskedasticity and serial correlation)')
    return params, bse, pvals


print('Building annual panel')
ann_panel = build_panel(FY_WINDOWS, valid, factors_d, cg_ann_map, window_type = 'fy')
print(f'Annual panel: {len(ann_panel):,} obs, {ann_panel["NSE Symbol"].nunique()} firms, '
      f'{ann_panel["window"].nunique()} FYs')
run_panel_ols(ann_panel, 'FF5 + MOM + CG — Annual FY Panel (pooled)')

print('\nBuilding filing-date panel')
filing_windows = [(row['NSE Symbol'], row['Q_FY'], row['Filing_Date'])
                  for _, row in q4_filings.iterrows()]
fil_panel = build_panel(filing_windows, valid, factors_d, cg_q_map, window_type = 'filing')
print(f'Filing panel: {len(fil_panel):,} obs, {fil_panel["NSE Symbol"].nunique()} firms, '
      f'{fil_panel["window"].nunique()} Q4s')
run_panel_ols(fil_panel, 'FF5 + MOM + CG — Filing-Date Q4 Panel (pooled)')

Annual CG composite:    1309 firm-FY pairs, score range: 0.20 – 0.73
Quarterly CG composite: 4535 firm-Q pairs
Building annual panel


Annual panel: 156,997 obs, 233 firms, 3 FYs

════════════════════════════════════════════════════════════
  FF5 + MOM + CG — Annual FY Panel (pooled)
  N firms: 233  |  N obs: 156,997  |  R²=0.0109  |  Adj-R²=0.0109
════════════════════════════════════════════════════════════
  Variable               Coef         SE        t        p  
  ────────────────────────────────────────────────────────────
  Mkt_RF             1.054501   0.030460   34.619   0.0000 ***
  SMB                0.368213   0.049579    7.427   0.0000 ***
  HML               -0.070012   0.043046   -1.626   0.1052 
  RMW               -0.026255   0.062331   -0.421   0.6740 
  CMA                0.171795   0.077918    2.205   0.0285 **
  MOM                0.082958   0.031180    2.661   0.0083 ***
  CG_score           0.000012   0.000736    0.016   0.9870  ◀
  ────────────────────────────────────────────────────────────
  const              0.000146
  Window FE: Yes (3 windows)
════════════════════════════════════════════

  JB p=0.0000  |  BP p=0.0000  |  DW=1.983
  (clustered SE corrects for heteroskedasticity and serial correlation)

Building filing-date panel


Filing panel: 165,988 obs, 233 firms, 3 Q4s

════════════════════════════════════════════════════════════
  FF5 + MOM + CG — Filing-Date Q4 Panel (pooled)
  N firms: 233  |  N obs: 165,988  |  R²=0.0099  |  Adj-R²=0.0099
════════════════════════════════════════════════════════════
  Variable               Coef         SE        t        p  
  ────────────────────────────────────────────────────────────
  Mkt_RF             1.043251   0.031496   33.123   0.0000 ***
  SMB                0.412501   0.044897    9.188   0.0000 ***
  HML               -0.062543   0.042264   -1.480   0.1403 
  RMW               -0.041120   0.059825   -0.687   0.4926 
  CMA                0.145011   0.075125    1.930   0.0548 *
  MOM                0.074196   0.028831    2.573   0.0107 **
  CG_score          -0.000345   0.000656   -0.527   0.5988  ◀
  ────────────────────────────────────────────────────────────
  const              0.000337
  Window FE: Yes (3 windows)
═════════════════════════════════════════

(const        0.000337
 Mkt_RF       1.043251
 SMB          0.412501
 HML         -0.062543
 RMW         -0.041120
 CMA          0.145011
 MOM          0.074196
 CG_score    -0.000345
 _w_Q4FY23    0.000072
 _w_Q4FY24   -0.000024
 dtype: float64,
 const        0.000335
 Mkt_RF       0.031496
 SMB          0.044897
 HML          0.042264
 RMW          0.059825
 CMA          0.075125
 MOM          0.028831
 CG_score     0.000656
 _w_Q4FY23    0.000132
 _w_Q4FY24    0.000142
 dtype: float64,
 const        3.161695e-01
 Mkt_RF       6.647193e-90
 SMB          2.331597e-17
 HML          1.402785e-01
 RMW          4.925592e-01
 CMA          5.479259e-02
 MOM          1.069146e-02
 CG_score     5.988106e-01
 _w_Q4FY23    5.866306e-01
 _w_Q4FY24    8.680525e-01
 dtype: float64)